<a href="https://colab.research.google.com/github/shaan-byte/python_ds_colab/blob/main/Session_5_2_Vectorized_Operations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 5.2 — Vectorized Operations

**Module 2: Data Analysis Foundations**  
**Week 5 | Session 5.2**

---

## What we will cover today

1. A quick recap — what "vectorized" means
2. Element-wise operations — going deeper
3. Broadcasting — operating on arrays of different shapes
4. Aggregations — sum, mean, standard deviation, and what they tell you
5. Masking — filtering and modifying data without loops
6. Combining everything into a real analysis

---

> **Where we left off:** In Session 5.1 we learned why NumPy is fast and how to create, shape, and inspect arrays. We briefly touched element-wise math and boolean filtering. Today we go deeper into exactly those two ideas, and add the one major concept we have not yet covered: **broadcasting** — how NumPy lets arrays of different shapes work together in a single operation, no loop required.

---
## Setup — Run this first

In [ ]:
import numpy as np

print("NumPy version:", np.__version__)
print("Ready.")

NumPy version: 2.0.2
Ready.


---
# Part 1 — Quick Recap: What Does "Vectorized" Mean?

**Vectorized** means an operation is applied to an entire array at once, with no explicit Python loop. NumPy handles the looping internally, using optimised C code.

```python
# NOT vectorized — a Python loop
result = []
for x in scores:
    result.append(x * 2)

# Vectorized — no loop
result = scores * 2
```

Today's session is entirely about mastering this "no-loop" style of thinking. By the end, when you see a problem that would normally need a `for` loop, you should be able to ask: **"Can NumPy do this in one vectorized expression instead?"** — and usually the answer is yes.

We continue with our IPL franchise analyst role from last session.

In [ ]:
72 / 48

1.5

In [ ]:
# Our squad's batting data: 5 players x 14 matches (same shape as last session)

PLAYERS = ["Virat", "Rohit", "Hardik", "Shubman", "Shreyas"]

squad_scores = np.array([
    [72, 45, 88, 31, 64,  0, 103, 52, 19, 77, 41, 93, 28, 66],   # Virat
    [56,  0, 74, 88, 32, 45,  61, 17, 90, 38, 72,  5, 83, 49],   # Rohit
    [45, 32,  0, 67, 18, 54,  78, 22, 41, 33, 60, 15, 88, 27],   # Hardik
    [33, 61, 45,  0, 77, 83,  22, 58, 44, 71,  0, 66, 31, 52],   # Shubman
    [88, 14, 57, 72,  0, 38,  91, 43, 66, 22, 84,  0, 47, 73],   # Shreyas
])

# Balls faced per player per match (needed later for strike rate)
balls_faced = np.array([
    [48, 32, 55, 24, 41,  0, 61, 38, 16, 50, 29, 58, 22, 44],
    [39,  0, 50, 56, 25, 33,  42, 14, 58, 27, 49,  6, 54, 35],
    [31, 24,  0, 46, 15, 38,  51, 18, 30, 26, 42, 12, 56, 21],
    [25, 42, 33,  0, 52, 56,  18, 40, 32, 47,  0, 44, 24, 36],
    [56, 12, 39, 48,  0, 28,  60, 31, 44, 17, 53,  0, 34, 47],
])

print(f"squad_scores shape: {squad_scores.shape}")
print(f"balls_faced shape:  {balls_faced.shape}")
print()
print("Both arrays have the same shape — this matters for what comes next.")

squad_scores shape: (5, 14)
balls_faced shape:  (5, 14)

Both arrays have the same shape — this matters for what comes next.


---
# Part 2 — Element-wise Operations, Properly

## The rule

An element-wise operation takes two arrays of the **same shape** and combines them position by position. Position `[i, j]` in the result comes only from position `[i, j]` in each input.

```
squad_scores[1, 4]   <-- Rohit's runs in match 5
balls_faced[1, 4]    <-- Rohit's balls faced in match 5

strike_rate[1, 4] = squad_scores[1, 4] / balls_faced[1, 4] * 100
```

This relationship holds for **every single position** in the array, computed all at once.

In [ ]:
(squad_scores / balls_faced) * 100

/tmp/ipykernel_9976/2178055193.py:1: RuntimeWarning: invalid value encountered in divide
  (squad_scores / balls_faced) * 100


array([[150.        , 140.625     , 160.        , 129.16666667,
        156.09756098,          nan, 168.85245902, 136.84210526,
        118.75      , 154.        , 141.37931034, 160.34482759,
        127.27272727, 150.        ],
       [143.58974359,          nan, 148.        , 157.14285714,
        128.        , 136.36363636, 145.23809524, 121.42857143,
        155.17241379, 140.74074074, 146.93877551,  83.33333333,
        153.7037037 , 140.        ],
       [145.16129032, 133.33333333,          nan, 145.65217391,
        120.        , 142.10526316, 152.94117647, 122.22222222,
        136.66666667, 126.92307692, 142.85714286, 125.        ,
        157.14285714, 128.57142857],
       [132.        , 145.23809524, 136.36363636,          nan,
        148.07692308, 148.21428571, 122.22222222, 145.        ,
        137.5       , 151.06382979,          nan, 150.        ,
        129.16666667, 144.44444444],
       [157.14285714, 116.66666667, 146.15384615, 150.        ,
                 nan

In [ ]:
# Strike rate = (runs / balls faced) * 100
# This needs element-wise division between two same-shaped arrays

# Problem: division by zero where a player didn't bat (0 balls faced)
# Let's see what NumPy does

with np.errstate(invalid='ignore'):
    strike_rate = (squad_scores / balls_faced) * 100

print("Strike rate (runs per 100 balls):")
print(strike_rate.round(1))
print()
print("Notice the 'nan' values — those are 0/0 (did not bat that match).")
print("NumPy does not crash on division by zero — it returns nan or inf and warns you.")

Strike rate (runs per 100 balls):
[[150.  140.6 160.  129.2 156.1   nan 168.9 136.8 118.8 154.  141.4 160.3
  127.3 150. ]
 [143.6   nan 148.  157.1 128.  136.4 145.2 121.4 155.2 140.7 146.9  83.3
  153.7 140. ]
 [145.2 133.3   nan 145.7 120.  142.1 152.9 122.2 136.7 126.9 142.9 125.
  157.1 128.6]
 [132.  145.2 136.4   nan 148.1 148.2 122.2 145.  137.5 151.1   nan 150.
  129.2 144.4]
 [157.1 116.7 146.2 150.    nan 135.7 151.7 138.7 150.  129.4 158.5   nan
  138.2 155.3]]

Notice the 'nan' values — those are 0/0 (did not bat that match).
NumPy does not crash on division by zero — it returns nan or inf and warns you.


In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 6, 7])
a

array([1, 2, 3])

In [ ]:
np.where((a + b) % 2 == 0, a, b)

array([4, 2, 3])

In [ ]:
a

array([1, 2, 3])

In [ ]:
np.maximum(a, b, a)

array([4, 6, 7])

In [ ]:
# A cleaner way: use np.where() to handle the zero-balls case explicitly
# np.where(condition, value_if_true, value_if_false) — itself fully vectorized

strike_rate_clean = np.where(
    balls_faced > 0,                       # condition, checked element-wise
    (squad_scores / np.maximum(balls_faced, 1)) * 100,   # avoid division by zero
    0                                       # value to use where condition is False
)

print("Strike rate (cleaned, 0 where player did not bat):")
print(strike_rate_clean.round(1))
print()
print("np.where() is itself a vectorized operation — no loop, even though")
print("it behaves like an if/else applied to every element.")

Strike rate (cleaned, 0 where player did not bat):
[[150.  140.6 160.  129.2 156.1   0.  168.9 136.8 118.8 154.  141.4 160.3
  127.3 150. ]
 [143.6   0.  148.  157.1 128.  136.4 145.2 121.4 155.2 140.7 146.9  83.3
  153.7 140. ]
 [145.2 133.3   0.  145.7 120.  142.1 152.9 122.2 136.7 126.9 142.9 125.
  157.1 128.6]
 [132.  145.2 136.4   0.  148.1 148.2 122.2 145.  137.5 151.1   0.  150.
  129.2 144.4]
 [157.1 116.7 146.2 150.    0.  135.7 151.7 138.7 150.  129.4 158.5   0.
  138.2 155.3]]

np.where() is itself a vectorized operation — no loop, even though
it behaves like an if/else applied to every element.


In [ ]:
# Element-wise comparison between two players
# Who scored more, Virat or Rohit, in each match?

virat = squad_scores[0]   # row 0
rohit = squad_scores[1]   # row 1

virat_scored_more = virat > rohit

print("Match-by-match comparison (Virat vs Rohit):")
for i in range(14):
    winner = "Virat" if virat_scored_more[i] else ("Rohit" if rohit[i] > virat[i] else "Tied")
    print(f"  Match {i+1:>2}: Virat={virat[i]:>3}  Rohit={rohit[i]:>3}  -> {winner}")

print()
print(f"Virat outscored Rohit in {virat_scored_more.sum()} of 14 matches")

---
### Exercise 1 — Element-wise Operations

**Task:** Complete the calculations below. Each should be a single vectorized expression — no loops.

In [ ]:
# Q1: Total runs scored by Virat and Rohit COMBINED, match by match
#     (element-wise addition of their two rows)
combined_runs = squad_scores[0] + squad_scores[___]   # <-- which row index for Rohit?

# Q2: The difference in balls faced between Hardik and Shubman, match by match
ball_diff = balls_faced[2] - balls_faced[___]          # <-- which row index for Shubman?

# Q3: A boolean array — True where Shreyas scored more than Hardik
shreyas_better = squad_scores[4] ___ squad_scores[2]   # <-- which comparison operator?

# Q4: A "bonus runs" array — every player's score with 5 bonus runs added
#     for matches where they scored a half-century (>= 50)
#     Hint: use np.where(condition, value_if_true, value_if_false)
bonus_scores = np.where(squad_scores >= ___, squad_scores + ___, squad_scores)


print(f"Q1 Combined Virat+Rohit: {combined_runs}")
print(f"Q2 Ball diff Hardik-Shubman: {ball_diff}")
print(f"Q3 Shreyas > Hardik: {shreyas_better}")
print(f"Q4 Bonus scores (Virat row): {bonus_scores[0]}")
print()
print(f"Correct Q1 (14 values):  {len(combined_runs) == 14}")
print(f"Correct Q3 (bool array): {shreyas_better.dtype == bool}")
print(f"Correct Q4 (bonus applied): {bonus_scores[0][2] == 93}")  # Virat's match 3 score 88 -> 93

<details>
<summary>Show solution</summary>

```python
combined_runs   = squad_scores[0] + squad_scores[1]
ball_diff       = balls_faced[2] - balls_faced[3]
shreyas_better  = squad_scores[4] > squad_scores[2]
bonus_scores    = np.where(squad_scores >= 50, squad_scores + 5, squad_scores)
```

</details>

---
# Part 3 — Broadcasting

## The problem broadcasting solves

Element-wise operations require both arrays to have the **same shape**. But very often, you want to combine a big array with a smaller one — for example:

- Add a fixed bonus of 10 runs to every score in the entire `(5, 14)` squad array
- Apply a different adjustment to each player (5 values) across all 14 matches
- Apply a different adjustment to each match (14 values) across all 5 players

None of these involve two arrays of the same shape. **Broadcasting** is the set of rules NumPy uses to "stretch" smaller arrays so they fit a larger one, without actually copying any data in memory.

---

## Broadcasting rule 1: scalar with array

You already used this in Session 5.1 without naming it:

```python
squad_scores + 10
```

The scalar `10` is broadcast to match every position in the `(5, 14)` array. This is the simplest case of broadcasting.

In [ ]:
# Scalar broadcasting — the simplest case

print("Original shape:", squad_scores.shape)
print()

bonus_flat = squad_scores + 10
print("squad_scores + 10  (every value gets +10):")
print(bonus_flat)
print()
print("The scalar 10 was 'broadcast' to all 70 positions (5 x 14) automatically.")

Original shape: (5, 14)

squad_scores + 10  (every value gets +10):
[[ 82  55  98  41  74  10 113  62  29  87  51 103  38  76]
 [ 66  10  84  98  42  55  71  27 100  48  82  15  93  59]
 [ 55  42  10  77  28  64  88  32  51  43  70  25  98  37]
 [ 43  71  55  10  87  93  32  68  54  81  10  76  41  62]
 [ 98  24  67  82  10  48 101  53  76  32  94  10  57  83]]

The scalar 10 was 'broadcast' to all 70 positions (5 x 14) automatically.


## Broadcasting rule 2: 1D array with 2D array

This is where broadcasting gets genuinely useful. Suppose each match had a different **pitch difficulty multiplier** — a single value per match, applied to every player's score in that match.

```
squad_scores shape:        (5, 14)   <- 5 players, 14 matches
pitch_multiplier shape:       (14,)  <- one value per match
```

NumPy compares shapes from the **right side**. `(5, 14)` vs `(14,)` — the trailing dimensions (`14` and `14`) match. So NumPy "stretches" the 1D array across all 5 rows, as if it had been repeated 5 times.

```
Conceptually, pitch_multiplier (14,) becomes:

[[m1, m2, m3, ..., m14],     <- repeated for row 0
 [m1, m2, m3, ..., m14],     <- repeated for row 1
 [m1, m2, m3, ..., m14],     <- repeated for row 2
 [m1, m2, m3, ..., m14],     <- repeated for row 3
 [m1, m2, m3, ..., m14]]     <- repeated for row 4
```

No actual copying happens in memory — NumPy is just clever about reusing the same values — but conceptually this is exactly what happens.

In [ ]:
marks = np.array([
    [10, 11, 15], # student 1 (sci, math and eng)
    [11, 14, 12], # student 2 (sci, math and eng)
    [10, 11, 15], # student 1 (sci, math and eng)
    [11, 14, 12] # student 2 (sci, math and eng)
])
marks
# diff max marks (sci - 15, math - 20, eng - 15 )

array([[10, 11, 15],
       [11, 14, 12],
       [10, 11, 15],
       [11, 14, 12]])

In [ ]:
for stud in marks:
  stud[0] * 100 / 15
  stud[1] * 100 / 20
  stud[2] * 100 / 15

In [ ]:
100 * marks

array([[1000, 1100, 1500],
       [1100, 1400, 1200],
       [1000, 1100, 1500],
       [1100, 1400, 1200]])

In [ ]:
max_marks = np.array([15, 20, 15])

In [ ]:
marks.shape

(2, 3)

In [ ]:
max_marks.shape

(3,)

In [ ]:
percentages = (marks * 100) / max_marks
percentages

array([[ 66.66666667,  55.        , 100.        ],
       [ 73.33333333,  70.        ,  80.        ],
       [ 66.66666667,  55.        , 100.        ],
       [ 73.33333333,  70.        ,  80.        ]])

In [ ]:
good_standing_students = np.array([1, 0, 1])

In [ ]:
print(good_standing_students.shape)
percentages.shape

(3,)


(4, 3)

In [ ]:
for i in range(percentages):
  if good_standing_students[i] == 0:
    percentages[i] = [0, 0, 0]

In [ ]:
print(good_standing_students.shape)
good_standing_students = good_standing_students.reshape(-1, 1)
good_standing_students.shape

(4,)


(4, 1)

In [ ]:
good_standing_students.reshape(2, 2)

array([[1, 0],
       [1, 1]])

In [ ]:
percentages.shape

(4, 3)

In [ ]:
percentages.reshape(12,)

array([ 66.66666667,  55.        , 100.        ,  73.33333333,
        70.        ,  80.        ,  66.66666667,  55.        ,
       100.        ,  73.33333333,  70.        ,  80.        ])

In [ ]:
percentages * good_standing_students

array([[ 66.66666667,  55.        , 100.        ],
       [  0.        ,   0.        ,   0.        ],
       [ 66.66666667,  55.        , 100.        ],
       [ 73.33333333,  70.        ,  80.        ]])

In [ ]:
mult = [1.1, 0.9, 1.0, 0.8, 1.2, 1.0, 0.95, 1.05, 1.0, 0.85, 1.15, 1.0, 0.9, 1.1]
scores = [
    [72, 45, 88, 31, 64,  0, 103, 52, 19, 77, 41, 93, 28, 66],   # Virat
    [56,  0, 74, 88, 32, 45,  61, 17, 90, 38, 72,  5, 83, 49],   # Rohit
    [45, 32,  0, 67, 18, 54,  78, 22, 41, 33, 60, 15, 88, 27],   # Hardik
    [33, 61, 45,  0, 77, 83,  22, 58, 44, 71,  0, 66, 31, 52],   # Shubman
    [88, 14, 57, 72,  0, 38,  91, 43, 66, 22, 84,  0, 47, 73],   # Shreyas
]

In [ ]:
new_scores = []
for player in scores:
  new_player = []
  for i in range(len(mult)):
    m = mult[i]
    scr = player[i]
    new_player.append(m * scr)
  new_scores.append(new_player)
print(new_scores)

[[79.2, 40.5, 88.0, 24.8, 76.8, 0.0, 97.85, 54.6, 19.0, 65.45, 47.15, 93.0, 25.2, 72.60000000000001], [61.60000000000001, 0.0, 74.0, 70.4, 38.4, 45.0, 57.949999999999996, 17.85, 90.0, 32.3, 82.8, 5.0, 74.7, 53.900000000000006], [49.50000000000001, 28.8, 0.0, 53.6, 21.599999999999998, 54.0, 74.1, 23.1, 41.0, 28.05, 69.0, 15.0, 79.2, 29.700000000000003], [36.300000000000004, 54.9, 45.0, 0.0, 92.39999999999999, 83.0, 20.9, 60.900000000000006, 44.0, 60.35, 0.0, 66.0, 27.900000000000002, 57.2], [96.80000000000001, 12.6, 57.0, 57.6, 0.0, 38.0, 86.45, 45.15, 66.0, 18.7, 96.6, 0.0, 42.300000000000004, 80.30000000000001]]


In [ ]:
# Pitch difficulty multiplier — one value per match (14 values)
# >1.0 means easier pitch (scores get boosted for context), <1.0 means harder pitch

pitch_multiplier = np.array([1.1, 0.9, 1.0, 0.8, 1.2, 1.0, 0.95, 1.05, 1.0, 0.85, 1.15, 1.0, 0.9])

print(f"squad_scores shape:    {squad_scores.shape}")
print(f"pitch_multiplier shape: {pitch_multiplier.shape}")
print()

adjusted_scores = squad_scores * pitch_multiplier

print("Adjusted scores (each match's column scaled by its pitch multiplier):")
print(adjusted_scores.round(1))
print()
print("Notice: every player's score in match 1 (column 0) was multiplied by 1.1")
print("        every player's score in match 4 (column 3) was multiplied by 0.8")
print(f"\nVirat match 1: {squad_scores[0,0]} x {pitch_multiplier[0]} = {adjusted_scores[0,0]:.1f}")
print(f"Rohit match 1: {squad_scores[1,0]} x {pitch_multiplier[0]} = {adjusted_scores[1,0]:.1f}")

squad_scores shape:    (5, 14)
pitch_multiplier shape: (13,)



ValueError: operands could not be broadcast together with shapes (5,14) (13,) 

## Broadcasting rule 3: per-row adjustment (the tricky one)

Now suppose instead we want a **per-player** adjustment — one value per player (5 values), applied across all 14 matches for that player.

```
squad_scores shape:           (5, 14)
player_adjustment shape:         (5,)
```

If we try to broadcast `(5,)` directly against `(5, 14)`, comparing from the right: `14` vs `5` — **these do not match and are not 1**. This will fail.

The fix: reshape the 1D array of 5 values into a **column** — shape `(5, 1)` — using `.reshape(-1, 1)` or by indexing with `[:, np.newaxis]`.

```
squad_scores shape:        (5, 14)
player_adjustment shape:    (5, 1)   <- now a column
```

Comparing from the right: `14` vs `1` (1 always broadcasts) — OK. Then `5` vs `5` — match. This works.

In [ ]:
squad_scores * player_form_rating

ValueError: operands could not be broadcast together with shapes (5,14) (5,) 

In [ ]:
# First, let's see the failure, so the fix makes sense

player_form_rating = np.array([1.15, 1.0, 0.95, 1.05, 1.1])   # one value per player (5,)

print(f"squad_scores shape:        {squad_scores.shape}")
print(f"player_form_rating shape:  {player_form_rating.shape}")
print()

try:
    bad_result = squad_scores * player_form_rating
    print(bad_result)
except ValueError as e:
    print(f"ERROR: {e}")
    print()
    print("NumPy compares shapes from the right: 14 vs 5 — neither matches, neither is 1.")
    print("Broadcasting fails. We need to reshape player_form_rating into a column.")

squad_scores shape:        (5, 14)
player_form_rating shape:  (5,)

ERROR: operands could not be broadcast together with shapes (5,14) (5,) 

NumPy compares shapes from the right: 14 vs 5 — neither matches, neither is 1.
Broadcasting fails. We need to reshape player_form_rating into a column.


In [ ]:
# The fix: reshape into a column vector (5, 1)

player_form_column = player_form_rating.reshape(-1, 1)   # -1 means "figure out this dimension"

print(f"player_form_rating shape:        {player_form_rating.shape}")
print(f"player_form_column shape:        {player_form_column.shape}")
print()
print("player_form_column looks like:")
print(player_form_column)
print()

form_adjusted_scores = squad_scores * player_form_column

print("Form-adjusted scores (each player's row scaled by their own form rating):")
print(form_adjusted_scores.round(1))
print()
print(f"Virat (row 0) form rating: {player_form_rating[0]}  ->  every value in row 0 x {player_form_rating[0]}")
print(f"  Check: {squad_scores[0,0]} x {player_form_rating[0]} = {form_adjusted_scores[0,0]:.2f}")

player_form_rating shape:        (5,)
player_form_column shape:        (5, 1)

player_form_column looks like:
[[1.15]
 [1.  ]
 [0.95]
 [1.05]
 [1.1 ]]

Form-adjusted scores (each player's row scaled by their own form rating):
[[ 82.8  51.7 101.2  35.6  73.6   0.  118.4  59.8  21.8  88.6  47.2 107.
   32.2  75.9]
 [ 56.    0.   74.   88.   32.   45.   61.   17.   90.   38.   72.    5.
   83.   49. ]
 [ 42.8  30.4   0.   63.6  17.1  51.3  74.1  20.9  38.9  31.4  57.   14.2
   83.6  25.6]
 [ 34.6  64.   47.2   0.   80.9  87.2  23.1  60.9  46.2  74.6   0.   69.3
   32.6  54.6]
 [ 96.8  15.4  62.7  79.2   0.   41.8 100.1  47.3  72.6  24.2  92.4   0.
   51.7  80.3]]

Virat (row 0) form rating: 1.15  ->  every value in row 0 x 1.15
  Check: 72 x 1.15 = 82.80


In [ ]:
form_adjusted_scores

array([[ 82.8 ,  51.75, 101.2 ,  35.65,  73.6 ,   0.  , 118.45,  59.8 ,
         21.85,  88.55,  47.15, 106.95,  32.2 ,  75.9 ],
       [ 56.  ,   0.  ,  74.  ,  88.  ,  32.  ,  45.  ,  61.  ,  17.  ,
         90.  ,  38.  ,  72.  ,   5.  ,  83.  ,  49.  ],
       [ 42.75,  30.4 ,   0.  ,  63.65,  17.1 ,  51.3 ,  74.1 ,  20.9 ,
         38.95,  31.35,  57.  ,  14.25,  83.6 ,  25.65],
       [ 34.65,  64.05,  47.25,   0.  ,  80.85,  87.15,  23.1 ,  60.9 ,
         46.2 ,  74.55,   0.  ,  69.3 ,  32.55,  54.6 ],
       [ 96.8 ,  15.4 ,  62.7 ,  79.2 ,   0.  ,  41.8 , 100.1 ,  47.3 ,
         72.6 ,  24.2 ,  92.4 ,   0.  ,  51.7 ,  80.3 ]])

### Whiteboard: the broadcasting rules, summarised

NumPy compares two array shapes starting from the **rightmost** dimension and moving left. At each position, the dimensions are compatible if:

1. They are **equal**, or
2. One of them is **1**

If every pair of dimensions satisfies one of these, broadcasting succeeds. If any pair fails both conditions, you get a `ValueError`.

```
  squad_scores        (5, 14)
  scalar                 ()      ->  treated as compatible with anything
  per-match           (14,)      ->  compare:        14 vs 14  OK
  per-player        (5, 1)       ->  compare:  5 vs 5,  1 vs 14 (1 stretches)  OK
  per-player (wrong)   (5,)      ->  compare:        5 vs 14  FAIL
```

**The most common beginner mistake:** trying to broadcast a `(5,)` array against a `(5, 14)` array, expecting it to apply per-row. It does not — NumPy always aligns from the right. You must reshape it to `(5, 1)` first.

**Mental shortcut:**
- A plain 1D array of length 14 broadcasts naturally **across rows** (one value per column).
- To broadcast **down columns** (one value per row), reshape to a column: `.reshape(-1, 1)`.

---
### Exercise 2 — Broadcasting

**Task:** Complete each broadcasting operation. Pay close attention to whether you need to reshape.

In [ ]:
flat_bonus = squad_scores + 3
flat_bonus

array([[ 75,  48,  91,  34,  67,   3, 106,  55,  22,  80,  44,  96,  31,
         69],
       [ 59,   3,  77,  91,  35,  48,  64,  20,  93,  41,  75,   8,  86,
         52],
       [ 48,  35,   3,  70,  21,  57,  81,  25,  44,  36,  63,  18,  91,
         30],
       [ 36,  64,  48,   3,  80,  86,  25,  61,  47,  74,   3,  69,  34,
         55],
       [ 91,  17,  60,  75,   3,  41,  94,  46,  69,  25,  87,   3,  50,
         76]])

In [ ]:
home_bonus = np.array([1.05]*7 + [1.0]*7)
home_adjusted = squad_scores * home_bonus

home_adjusted

array([[ 75.6 ,  47.25,  92.4 ,  32.55,  67.2 ,   0.  , 108.15,  52.  ,
         19.  ,  77.  ,  41.  ,  93.  ,  28.  ,  66.  ],
       [ 58.8 ,   0.  ,  77.7 ,  92.4 ,  33.6 ,  47.25,  64.05,  17.  ,
         90.  ,  38.  ,  72.  ,   5.  ,  83.  ,  49.  ],
       [ 47.25,  33.6 ,   0.  ,  70.35,  18.9 ,  56.7 ,  81.9 ,  22.  ,
         41.  ,  33.  ,  60.  ,  15.  ,  88.  ,  27.  ],
       [ 34.65,  64.05,  47.25,   0.  ,  80.85,  87.15,  23.1 ,  58.  ,
         44.  ,  71.  ,   0.  ,  66.  ,  31.  ,  52.  ],
       [ 92.4 ,  14.7 ,  59.85,  75.6 ,   0.  ,  39.9 ,  95.55,  43.  ,
         66.  ,  22.  ,  84.  ,   0.  ,  47.  ,  73.  ]])

In [ ]:
a = np.array([
    [1, 2, 3, 8],
    [4, 5, 6, 12],
    [7, 8, 10, 9]
])
a

array([[ 1,  2,  3,  8],
       [ 4,  5,  6, 12],
       [ 7,  8, 10,  9]])

In [ ]:
a.reshape(4, 3)

array([[ 1,  2,  3],
       [ 8,  4,  5],
       [ 6, 12,  7],
       [ 8, 10,  9]])

In [ ]:
a.reshape(4, -1)

array([[ 1,  2,  3],
       [ 8,  4,  5],
       [ 6, 12,  7],
       [ 8, 10,  9]])

In [ ]:
a.reshape(-1)

array([ 1,  2,  3,  8,  4,  5,  6, 12,  7,  8, 10,  9])

In [ ]:
a.reshape(-1, 1)

array([[ 1],
       [ 2],
       [ 3],
       [ 8],
       [ 4],
       [ 5],
       [ 6],
       [12],
       [ 7],
       [ 8],
       [10],
       [ 9]])

In [ ]:
# Q1: Add a flat bonus of 3 runs to EVERY score in the entire squad
#     (scalar broadcasting)
flat_bonus = squad_scores + 3

# Q2: A 'home advantage' multiplier — one value per match (14 values, all matches at home this time)
#     home_bonus shape is (14,). Apply it directly — no reshape needed.
home_bonus = np.array([1.05]*7 + [1.0]*7)   # first 7 matches at home, next 7 away
home_adjusted = squad_scores * home_bonus          # <-- which array? does it need reshaping?

# Q3: A 'captain bonus' — Virat (row 0) gets a personal +5 boost on every match,
#     everyone else gets +0. Build this as a (5, 1) column and broadcast it.
captain_bonus_flat = np.array([5, 0, 0, 0, 0])      # one value per player
captain_bonus_col  = captain_bonus_flat.reshape(-1, 1)  # <-- which method reshapes it to (5,1)?
captain_adjusted    = squad_scores + captain_bonus_col            # <-- which array to add?


print("Q1 flat_bonus row 0:       ", flat_bonus[0])
print("Q2 home_adjusted row 0:    ", home_adjusted[0].round(1))
print("Q3 captain_adjusted col 0: ", captain_adjusted[:, 0])
print()
print(f"Correct Q1 (every value +3):     {(flat_bonus[0] == squad_scores[0] + 3).all()}")
print(f"Correct Q2 shape preserved:      {home_adjusted.shape == (5, 14)}")
print(f"Correct Q3 (only Virat boosted): {captain_adjusted[0,0] == squad_scores[0,0]+5 and captain_adjusted[1,0] == squad_scores[1,0]}")

Q1 flat_bonus row 0:        [ 75  48  91  34  67   3 106  55  22  80  44  96  31  69]
Q2 home_adjusted row 0:     [ 75.6  47.2  92.4  32.6  67.2   0.  108.2  52.   19.   77.   41.   93.
  28.   66. ]
Q3 captain_adjusted col 0:  [77 56 45 33 88]

Correct Q1 (every value +3):     True
Correct Q2 shape preserved:      True
Correct Q3 (only Virat boosted): True


<details>
<summary>Show solution</summary>

```python
flat_bonus          = squad_scores + 3
home_adjusted        = squad_scores * home_bonus
captain_bonus_col    = captain_bonus_flat.reshape(-1, 1)
captain_adjusted     = squad_scores + captain_bonus_col
```

</details>

---
# Part 4 — Aggregations: Sum, Mean, Standard Deviation

## Recap from Session 5.1

We briefly used `.sum()` and `.mean()` last time. Today we go deeper, and add **standard deviation** — a measure of consistency that is essential for real analysis.

---

## Sum and Mean — totals and central tendency

In [ ]:
# Quick recap with axis

print("Sum and Mean:")
print(f"  Total runs, whole squad, whole season: {squad_scores.sum()}")
print(f"  Total runs per player (axis=1):        {squad_scores.sum(axis=1)}")
print(f"  Total runs per match  (axis=0):         {squad_scores.sum(axis=0)}")
print()
print(f"  Average score, whole squad:             {squad_scores.mean():.2f}")
print(f"  Average per player (axis=1):            {squad_scores.mean(axis=1).round(2)}")
print(f"  Average per match  (axis=0):            {squad_scores.mean(axis=0).round(2)}")

## Standard deviation — measuring consistency

**Standard deviation** measures how spread out values are from the mean. A low standard deviation means the values are clustered close to the average — consistent. A high standard deviation means the values are spread out — inconsistent.

### Why this matters for our cricket analysis

Two players can have the exact same average but very different reliability:

```
Player A: 50, 48, 52, 49, 51   ->  mean = 50,  std = very low   (consistent)
Player B: 0, 10, 100, 5, 135   ->  mean = 50,  std = very high  (boom or bust)
```

A team captain might prefer Player A for a tight chase, and Player B as a high-risk, high-reward option. Standard deviation is how you quantify that difference.

In [ ]:
# Demonstrating with the example above

player_a = np.array([50, 48, 52, 49, 51])
player_b = np.array([0, 10, 100, 5, 135])

print(f"Player A: {player_a}")
print(f"  mean = {player_a.mean():.1f}   std = {player_a.std():.2f}")
print()
print(f"Player B: {player_b}")
print(f"  mean = {player_b.mean():.1f}   std = {player_b.std():.2f}")
print()
print("Same mean, wildly different standard deviation.")
print("Player A is the reliable, consistent option.")
print("Player B is the unpredictable, high-variance option.")

In [ ]:
# Now apply this to our actual squad

means = squad_scores.mean(axis=1)
stds  = squad_scores.std(axis=1)

print("CONSISTENCY REPORT")
print("=" * 55)
print(f"  {'Player':<10} {'Average':>9}  {'Std Dev':>9}  {'Consistency'}")
print("  " + "-" * 50)

for i, player in enumerate(PLAYERS):
    consistency = "Very consistent" if stds[i] < 20 else ("Moderate" if stds[i] < 30 else "Inconsistent")
    print(f"  {player:<10} {means[i]:>9.1f}  {stds[i]:>9.1f}  {consistency}")

print()
most_consistent = PLAYERS[stds.argmin()]
least_consistent = PLAYERS[stds.argmax()]
print(f"Most consistent player:  {most_consistent} (std = {stds.min():.1f})")
print(f"Least consistent player: {least_consistent} (std = {stds.max():.1f})")

---
### Exercise 3 — Aggregations

**Task:** Using `squad_scores`, compute the following. Every answer should be one vectorized expression.

In [ ]:
# Q1: Standard deviation of scores for EACH MATCH (across all 5 players)
#     i.e. how spread out were the team's scores within match 1, match 2, etc.
match_std = squad_scores.std(axis=___)

# Q2: Which match had the most inconsistent scores across players?
#     Hint: use .argmax() on match_std
most_inconsistent_match = match_std.___() + 1   # <-- which method? (+1 converts to match number)

# Q3: The overall team standard deviation (single number, entire 2D array)
overall_std = squad_scores.___()

# Q4: Range (max - min) of scores for each player
#     Hint: there's no .range() method — compute it from .max() and .min()
player_range = squad_scores.max(axis=1) - squad_scores.___(axis=1)


print(f"Q1 Match std devs:           {match_std.round(1)}")
print(f"Q2 Most inconsistent match:  Match {most_inconsistent_match}")
print(f"Q3 Overall team std:         {overall_std:.2f}")
print(f"Q4 Player ranges:            {player_range}")
print()
print(f"Correct Q1 (14 values):  {len(match_std) == 14}")
print(f"Correct Q4 (5 values):   {len(player_range) == 5}")

<details>
<summary>Show solution</summary>

```python
match_std               = squad_scores.std(axis=0)
most_inconsistent_match = match_std.argmax() + 1
overall_std              = squad_scores.std()
player_range             = squad_scores.max(axis=1) - squad_scores.min(axis=1)
```

</details>

---
# Part 5 — Masking

## What is a mask?

A **mask** is a boolean array, the same shape as your data, where `True` marks positions you want to select and `False` marks positions you want to ignore. We saw the basics of this in Session 5.1 as "boolean indexing" — masking is the same idea, applied more deliberately and often in 2D.

```
data:  [72, 45, 88, 31, 64]
mask:  [ T,  F,  T,  F,  T]    (data > 50)

data[mask]  ->  [72, 88, 64]
```

Masking lets you select, count, and modify data based on a condition — all without writing a loop.

---

## Building masks

In [ ]:
# A mask is just a comparison — it returns an array of True/False

half_century_mask = squad_scores >= 50

print("squad_scores >= 50:")
print(half_century_mask)
print()
print(f"Mask shape: {half_century_mask.shape}  (same as squad_scores)")
print(f"Mask dtype: {half_century_mask.dtype}")
print()

# Counting True values — sum() treats True as 1, False as 0
total_fifties = half_century_mask.sum()
print(f"Total half-centuries across the whole squad: {total_fifties}")

fifties_per_player = half_century_mask.sum(axis=1)
print(f"Half-centuries per player: {fifties_per_player}")
for i, p in enumerate(PLAYERS):
    print(f"  {p}: {fifties_per_player[i]}")

In [ ]:
# Using a mask to SELECT values (extraction)

# All half-century scores, flattened into a 1D array (loses position info)
all_fifty_plus = squad_scores[half_century_mask]
print(f"All scores of 50+ across the whole squad: {all_fifty_plus}")
print(f"  (this is a flat 1D array — {len(all_fifty_plus)} values total)")
print()

# Selecting from a single row
virat_scores = squad_scores[0]
virat_fifties = virat_scores[virat_scores >= 50]
print(f"Virat's half-centuries: {virat_fifties}")
print()

# Combining conditions with & (and), | (or), ~ (not)
# IMPORTANT: use & and | , not 'and'/'or' — and wrap each condition in parentheses
good_but_not_great = squad_scores[(squad_scores >= 40) & (squad_scores < 70)]
print(f"Scores between 40 and 70 (whole squad): {good_but_not_great}")
print()

ducks_or_centuries = squad_scores[(squad_scores == 0) | (squad_scores >= 100)]
print(f"Ducks or centuries (the extremes): {ducks_or_centuries}")

In [ ]:
# Using a mask to MODIFY values in place
# This is different from selection — we assign new values to the masked positions

# Make a copy so we don't alter the original squad_scores
capped_scores = squad_scores.copy()

# Cap every score at 80 — anything above 80 becomes exactly 80
# (imagine simulating a "what if no one could score more than 80" scenario)
capped_scores[capped_scores > 80] = 80

print("Original Virat scores: ", squad_scores[0])
print("Capped Virat scores:   ", capped_scores[0])
print()
print("Every value above 80 was replaced with exactly 80 — no loop needed.")
print()

# A second example: zero out any score below 20 (treat as "failure")
performance_only = squad_scores.copy()
performance_only[performance_only < 20] = 0

print("Original Hardik scores:", squad_scores[2])
print("Failures zeroed out:   ", performance_only[2])

### Important: `.copy()` matters

When you write `new_array = old_array`, you do **not** get a new array — you get another name pointing to the same data. Modifying `new_array` would also modify `old_array`.

`.copy()` creates an actual independent array. Always use it when you want to modify a version of your data while keeping the original untouched — which is almost always what you want during analysis.

In [ ]:
# np.where() revisited — a mask-based way to build a NEW array
# without modifying the original

# Classify every score as 'duck', 'failure', 'starter', 'good', or 'fifty_plus'
# We do this with nested np.where() calls — innermost evaluated first

classification = np.where(
    squad_scores == 0, "duck",
    np.where(
        squad_scores < 20, "failure",
        np.where(
            squad_scores < 40, "starter",
            np.where(
                squad_scores < 50, "good",
                "fifty_plus"
            )
        )
    )
)

print("Classification of Virat's matches:")
for i in range(14):
    print(f"  Match {i+1:>2}: score={squad_scores[0,i]:>3}  ->  {classification[0,i]}")

---
### Exercise 4 — Masking

**Task:** Complete the analysis below using masks. Every step should avoid explicit loops.

In [ ]:
# Q1: A mask that is True wherever a player scored a 'duck' (score == 0)
duck_mask = squad_scores ___ 0

# Q2: Total number of ducks across the WHOLE squad
total_ducks = duck_mask.___()

# Q3: All scores between 60 and 90 (inclusive), extracted as a flat array
mid_range_scores = squad_scores[(squad_scores ___ 60) & (squad_scores ___ 90)]

# Q4: Make a copy of squad_scores, then replace every duck (0) with -1
#     to clearly mark it as a failure rather than confusing it with 'did not bat'
marked_scores = squad_scores.___()        # <-- how do we avoid modifying the original?
marked_scores[marked_scores == 0] = ___   # <-- what value?

# Q5: Count how many CENTURIES (>= 100) each player scored
centuries_per_player = (squad_scores ___ 100).sum(axis=___)


print(f"Q1 duck_mask shape:        {duck_mask.shape}")
print(f"Q2 total_ducks:            {total_ducks}")
print(f"Q3 mid_range_scores:       {mid_range_scores}")
print(f"Q4 marked_scores Hardik:   {marked_scores[2]}")
print(f"Q5 centuries_per_player:   {centuries_per_player}")
print()
print(f"Correct Q2 (8 ducks total): {total_ducks == 8}")
print(f"Correct Q4 (original unchanged): {(squad_scores[2] == [45, 32, 0, 67, 18, 54, 78, 22, 41, 33, 60, 15, 88, 27]).all()}")
print(f"Correct Q5 (Virat has 1 century): {centuries_per_player[0] == 1}")

<details>
<summary>Show solution</summary>

```python
duck_mask              = squad_scores == 0
total_ducks            = duck_mask.sum()
mid_range_scores       = squad_scores[(squad_scores >= 60) & (squad_scores <= 90)]
marked_scores          = squad_scores.copy()
marked_scores[marked_scores == 0] = -1
centuries_per_player    = (squad_scores >= 100).sum(axis=1)
```

</details>

---
# Part 6 — Putting It All Together

Let's build a complete analysis that uses broadcasting, aggregations, and masking together — a single workflow with zero explicit loops.

**The brief from the coach:** "I want a fitness-adjusted performance score for each player. Apply each player's fitness multiplier to their raw scores, then tell me their adjusted average, their consistency, and how many adjusted scores crossed 60."

In [ ]:
# Fitness multiplier per player (5 values) — represents current physical condition
fitness_multiplier = np.array([1.05, 0.95, 1.0, 1.1, 0.9])

# Step 1: Broadcasting — reshape to a column and apply per-player
fitness_column = fitness_multiplier.reshape(-1, 1)
adjusted = squad_scores * fitness_column

# Step 2: Aggregations — adjusted average and consistency per player
adjusted_avg = adjusted.mean(axis=1)
adjusted_std = adjusted.std(axis=1)

# Step 3: Masking — count adjusted scores that crossed 60
crossed_60 = (adjusted >= 60).sum(axis=1)

# Report
print("FITNESS-ADJUSTED PERFORMANCE REPORT")
print("=" * 65)
print(f"  {'Player':<10} {'Fitness':>8}  {'Adj. Avg':>9}  {'Adj. Std':>9}  {'60+ count':>10}")
print("  " + "-" * 55)

for i, p in enumerate(PLAYERS):
    print(f"  {p:<10} {fitness_multiplier[i]:>8.2f}  {adjusted_avg[i]:>9.1f}  {adjusted_std[i]:>9.1f}  {crossed_60[i]:>10}")

print()
top_adjusted = PLAYERS[adjusted_avg.argmax()]
print(f"Top performer after fitness adjustment: {top_adjusted} (avg {adjusted_avg.max():.1f})")

---
## Final Exercise — End-to-End Challenge

This exercise combines broadcasting, aggregations, and masking into one full analysis. No loops allowed.

**Scenario:** The coach wants a "clutch performance" report. A player is considered to have performed in the clutch if their match score is at or above their OWN personal average for the season. You need to:

1. Compute each player's personal average (one value per player)
2. Use broadcasting to compare every match score against that player's own average
3. Build a mask of "clutch" performances (score >= personal average)
4. Count how many clutch performances each player had
5. Compute the average score specifically during clutch performances vs non-clutch performances

In [ ]:
# Step 1: Each player's personal average (5,)
personal_avg = squad_scores.mean(axis=___)

print("Personal averages:")
for i, p in enumerate(PLAYERS):
    print(f"  {p}: {personal_avg[i]:.1f}")

In [ ]:
# Step 2: Broadcast personal_avg against squad_scores
# personal_avg has shape (5,) — squad_scores has shape (5, 14)
# Reshape personal_avg so it broadcasts per-row, not per-column

personal_avg_col = personal_avg.reshape(___, ___)   # <-- what shape do we need?

print(f"personal_avg shape:     {personal_avg.shape}")
print(f"personal_avg_col shape: {personal_avg_col.shape}")
print(f"squad_scores shape:     {squad_scores.shape}")

In [ ]:
# Step 3: Build the clutch mask
# True where a player's score in that match was >= their own personal average

clutch_mask = squad_scores ___ personal_avg_col   # <-- which comparison operator?

print("Clutch mask (Virat's row):")
print(clutch_mask[0])
print(f"(Virat's personal average is {personal_avg[0]:.1f} — True where score >= that)")

In [ ]:
# Step 4: Count clutch performances per player

clutch_count = clutch_mask.sum(axis=___)

print("Clutch performance counts:")
for i, p in enumerate(PLAYERS):
    print(f"  {p}: {clutch_count[i]} out of 14 matches")

In [ ]:
# Step 5: Average score during clutch vs non-clutch performances, PER PLAYER
# This requires masking each row individually since the mask shape differs per player
# (we cannot use a single vectorized call for a ragged result, so a short loop
#  over just 5 players — not 70 cells — is acceptable here)

print("CLUTCH PERFORMANCE REPORT")
print("=" * 70)
print(f"  {'Player':<10} {'Personal Avg':>13}  {'Clutch Avg':>11}  {'Non-Clutch Avg':>15}")
print("  " + "-" * 60)

for i, p in enumerate(PLAYERS):
    player_scores  = squad_scores[i]
    player_mask    = clutch_mask[___]              # <-- which row?

    clutch_scores     = player_scores[player_mask]
    non_clutch_scores = player_scores[~player_mask]  # <-- ~ means NOT

    clutch_avg     = clutch_scores.mean() if len(clutch_scores) > 0 else 0
    non_clutch_avg = non_clutch_scores.___() if len(non_clutch_scores) > 0 else 0   # <-- which method?

    print(f"  {p:<10} {personal_avg[i]:>13.1f}  {clutch_avg:>11.1f}  {non_clutch_avg:>15.1f}")

print()
print("Notice: clutch average is always >= personal average by definition.")
print("This confirms our mask was built correctly.")
print()
print(f"Correct (clutch_count sums to something sensible): {clutch_count.sum() > 0 and clutch_count.sum() < 70}")

<details>
<summary>Show solution</summary>

```python
# Step 1
personal_avg = squad_scores.mean(axis=1)

# Step 2
personal_avg_col = personal_avg.reshape(-1, 1)

# Step 3
clutch_mask = squad_scores >= personal_avg_col

# Step 4
clutch_count = clutch_mask.sum(axis=1)

# Step 5
player_mask    = clutch_mask[i]
non_clutch_avg = non_clutch_scores.mean() if len(non_clutch_scores) > 0 else 0
```

</details>

---
## Session Summary

### Element-wise operations
- Two arrays of the **same shape** combine position by position
- `np.where(condition, value_if_true, value_if_false)` is a vectorized if/else — use it to handle special cases (like division by zero) without loops
- Comparisons (`>`, `<`, `==`) between two same-shaped arrays produce a boolean array, element-wise

### Broadcasting
- Lets arrays of **different shapes** work together by "stretching" the smaller one
- NumPy compares shapes from the **right**: dimensions must be equal, or one of them must be 1
- A 1D array of length matching the last dimension broadcasts naturally across rows
- To broadcast per-row instead, reshape to a column: `.reshape(-1, 1)`
- Failed broadcasts raise a clear `ValueError` — read it carefully, it tells you the shapes involved

### Aggregations
- `.sum()`, `.mean()` — totals and central tendency, with `axis=0` (down columns) or `axis=1` (across rows)
- `.std()` — standard deviation, measures consistency. Low = reliable, high = volatile
- Same mean does not mean same behaviour — always check standard deviation alongside the average

### Masking
- A mask is a boolean array the same shape as your data
- `array[mask]` **selects** matching values (returns a flat 1D array of just the matches)
- `array[mask] = value` **modifies** matching positions in place
- Combine conditions with `&` (and), `|` (or), `~` (not) — always wrap each condition in parentheses
- Always use `.copy()` before modifying an array if you want to preserve the original
- `mask.sum()` counts `True` values (since `True` behaves as `1`)

---

### What comes next

You now have the core vectorized toolkit: element-wise math, broadcasting, aggregation, and masking. These four ideas are the foundation of nearly everything in Pandas, which we move to next — every Pandas Series and DataFrame column is built on exactly these NumPy mechanics, just with row and column labels added on top.

---

### Quick reference

```python
import numpy as np

# Element-wise
a + b                              # same-shape arrays, position by position
np.where(cond, val_true, val_false)  # vectorized if/else

# Broadcasting
arr + scalar                       # scalar broadcasts everywhere
arr_2d * arr_1d                    # 1D broadcasts across rows (matches last dim)
arr_2d * arr_1d.reshape(-1, 1)      # reshaped 1D broadcasts down columns (per-row)

# Aggregations
arr.sum(axis=0)    # down columns
arr.sum(axis=1)    # across rows
arr.mean()
arr.std()          # standard deviation

# Masking
mask = arr > 50                    # boolean array, same shape
arr[mask]                          # select matching values (flat)
arr[mask] = 0                      # modify matching values in place
arr[(arr > 10) & (arr < 50)]       # combine conditions
mask.sum()                         # count True values
arr.copy()                         # independent copy before modifying
```